In [1]:
#Parameters
ticker = "AAPL"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "data/data.db"

In [2]:
# Parameters
ticker = "AAPL"
start_date = "2024-06-01"
end_date = "2024-11-30"
db_path = "C:\\Users\\shrav\\ucb_mfe\\UCB-MFE-python-preprogram-2026-winter\\Homeworks\\HW2\\Shravani_Koli\\data\\data.db"


In [3]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Connect to SQLite
conn = sqlite3.connect(db_path)
# Load cleaned stock data 
query = """
SELECT ts, open, high, low, close, adj_close, volume, ticker
FROM ohlc_hw_cleaned
WHERE ticker = ?
  AND ts >= ?
  AND ts <= ?
ORDER BY ts
"""

stock_df = pd.read_sql_query(query, conn, params=(ticker, start_date, end_date),parse_dates=["ts"])
conn.close()
print("Ticker:", ticker)
print("Number of records:", len(stock_df))
print("Date range:", start_date, "to", end_date)

stock_df.head()

Ticker: AAPL
Number of records: 90
Date range: 2024-06-01 to 2024-11-30


,ts,open,high,low,close,adj_close,volume,ticker
0,2024-06-03,192.90,194.99,192.52,194.03,193.81,50080539,AAPL
1,2024-06-04,194.64,195.32,193.03,194.35,194.13,47471445,AAPL
2,2024-06-05,195.40,196.90,194.87,195.87,195.64,54156785,AAPL
3,2024-06-06,195.69,196.50,194.17,194.48,194.26,41181753,AAPL
4,2024-06-07,194.65,196.94,194.14,196.89,196.66,53103912,AAPL


In [4]:
#daily logarithmic returns using adjusted closing prices
stock_df["log_return"] = np.log(stock_df["adj_close"] / stock_df["adj_close"].shift(1))

# Calculate 20-day rolling volatility, annualized
stock_df["rolling_volatility"] = (stock_df["log_return"].rolling(window=20).std()* np.sqrt(252))

#cumulative returns
stock_df["cumulative_return"] = (stock_df["adj_close"] / stock_df["adj_close"].iloc[0])
#running peak
stock_df["running_peak"] = stock_df["cumulative_return"].cummax()
#calculate drawdown
stock_df["drawdown"] = (stock_df["cumulative_return"] / stock_df["running_peak"] - 1)

# the calculated columns
stock_df[
    ["ts", "adj_close", "log_return",
     "rolling_volatility", "cumulative_return", "drawdown"]
].head(25)

,ts,adj_close,log_return,rolling_volatility,cumulative_return,drawdown
0,2024-06-03,193.81,NaN,NaN,1.000000,0.000000
1,2024-06-04,194.13,0.001650,NaN,1.001651,0.000000
2,2024-06-05,195.64,0.007748,NaN,1.009442,0.000000
3,2024-06-06,194.26,-0.007079,NaN,1.002322,-0.007054
4,2024-06-07,196.66,0.012279,NaN,1.014705,0.000000
5,2024-06-10,192.90,-0.019304,NaN,0.995305,-0.019119
6,2024-06-11,206.91,0.070112,NaN,1.067592,0.000000
7,2024-06-12,212.82,0.028163,NaN,1.098086,0.000000
8,2024-06-13,213.99,0.005483,NaN,1.104123,0.000000
9,2024-06-14,212.24,-0.008212,NaN,1.095093,-0.008178


In [5]:
# Using 2% annual risk-free rate
risk_free_rate = 0.02
trading_days = 252
daily_returns = stock_df["log_return"].dropna()

#performance metrics
total_return = (stock_df["adj_close"].iloc[-1] /stock_df["adj_close"].iloc[0] - 1)

annualized_return = np.exp(daily_returns.mean() * trading_days) - 1
annualized_volatility = (daily_returns.std() * np.sqrt(trading_days))

sharpe_ratio = ((daily_returns.mean() * trading_days - risk_free_rate)/ annualized_volatility if annualized_volatility > 0 else np.nan)

max_drawdown = stock_df["drawdown"].min()
best_day = np.exp(daily_returns.max()) - 1
worst_day = np.exp(daily_returns.min()) - 1
positive_days = (daily_returns > 0).mean() * 100

avg_up_day = (np.exp(daily_returns[daily_returns > 0]) - 1).mean()

avg_down_day = (np.exp(daily_returns[daily_returns < 0]) - 1).mean()

#displaying the 10 metrics 
metrics = pd.DataFrame({
    "Metric": [
        "Total Return",
        "Annualized Return",
        "Annualized Volatility",
        "Sharpe Ratio",
        "Maximum Drawdown",
        "Best Day",
        "Worst Day",
        "Positive Days (%)",
        "Average Up Day",
        "Average Down Day"
    ],
    "Value": [
        total_return,
        annualized_return,
        annualized_volatility,
        sharpe_ratio,
        max_drawdown,
        best_day,
        worst_day,
        positive_days,
        avg_up_day,
        avg_down_day
    ]
})

metrics

,Metric,Value
0,Total Return,0.212528
1,Annualized Return,0.725717
2,Annualized Volatility,0.257752
3,Sharpe Ratio,2.039334
4,Maximum Drawdown,-0.117502
5,Best Day,0.072628
6,Worst Day,-0.041630
7,Positive Days (%),62.921348
8,Average Up Day,0.011326
9,Average Down Day,-0.013021


In [6]:
import scrapbook as sb

# Store the metrics so Papermill and Scrapbook can retrieve them
report_metrics = {
    "ticker": ticker,
    "avg_daily_return": float(daily_returns.mean()),
    "avg_volatility": float(stock_df["rolling_volatility"].mean()),
    "max_drawdown": float(max_drawdown),
    "sharpe_ratio": float(sharpe_ratio),
    "total_return": float(total_return)
}

sb.glue("report_metrics", report_metrics)

print("Report metrics saved for:", ticker)
print(report_metrics)

Report metrics saved for: AAPL
{'ticker': 'AAPL', 'avg_daily_return': 0.0021652496228258735, 'avg_volatility': 0.24386066421766966, 'max_drawdown': -0.1175015988062248, 'sharpe_ratio': 2.039334211549391, 'total_return': 0.21252773334709252}


In [7]:
#finding dates with the largest daily price movements
stock_df["daily_return"] = stock_df["adj_close"].pct_change()

extreme_days = stock_df.loc[stock_df["daily_return"].abs().nlargest(10).index,["ts", "adj_close", "daily_return"]]
extreme_days

,ts,adj_close,daily_return
6,2024-06-11,206.91,0.072628
39,2024-08-05,209.03,-0.041630
68,2024-09-19,228.87,0.037066
76,2024-10-01,226.21,-0.029142
18,2024-07-01,216.50,0.029090
7,2024-06-12,212.82,0.028563
66,2024-09-16,216.32,-0.027775
57,2024-09-03,222.77,-0.027205
32,2024-07-24,218.29,-0.025709
29,2024-07-17,228.62,-0.025282


In [8]:
#inspecting suspicious AAPL price records
stock_df.loc[
    stock_df["ts"].isin(
        pd.to_datetime([
            "2024-06-27",
            "2024-08-22",
            "2024-09-17"
        ])
    ),
    ["ts", "open", "high", "low", "close", "adj_close"]
]

,ts,open,high,low,close,adj_close
